### Human in the Loop (HITL)

source: langgraph Documentation 

The Human-in-the-Loop (HITL) middleware lets you add human oversight to agent tool calls. When a model proposes an action that might require review—for example, writing to a file or executing SQL—the middleware can pause execution and wait for a decision.

It does this by checking each tool call against a configurable policy. If intervention is needed, the middleware issues an interrupt that halts execution. 

The graph state is saved using LangGraph’s persistence layer, so execution can pause safely and resume later.

A human decision then determines what happens next: the action can be approved as-is (approve), modified before running (edit), rejected with feedback (reject), or responded to directly (respond) for “ask user” style tools.


#### Interrupt Decision Types
The middleware defines four built-in ways a human can respond to an interrupt:


| Decision Type | Description | Example Use Case |
|:---:|---|---|
| ✅ **approve** | Execute the tool with the original arguments as proposed by the agent. | Send an email draft exactly as written |
| ✏️ **edit** | Modify the tool arguments before execution. | Change the recipient before sending an email |
| ❌ **reject** | Skip executing this tool call entirely and return rejection feedback to the agent. | Deny file deletion and explain why |
| 💬 **respond** | Return the human's message directly as a synthetic tool result, skipping execution, for "ask user" style tools. | Answer an "ask_user" prompt with a direct reply |



The available decision types for each tool depend on the policy you configure in interrupt_on. When multiple tool calls are paused at the same time, each action requires a separate decision. Decisions must be provided in the same order as the actions appear in the interrupt request.

Use reject when the human is denying the requested action. Use respond only when the human is acting as the tool, such as answering an ask_user prompt. Do not use respond to deny side-effecting tools, because its message is treated as a successful tool result.

#### Why Human in the Loop Exists

LLM agents can hallucinate, misinterpret intent, or take irreversible actions (deleting records, sending emails, charging credit cards). Human-in-the-loop (HITL) is the mechanism that lets you pause a running graph, surface its current state to a human, and then either approve, reject, or modify that state before execution continues.

LangGraph implements HITL natively through:
1. Interrupts — pause execution at a node boundary
2. Resume — continue execution from where it paused
3. update_state() — inject human-modified state before resuming
4. Approval Workflows — structured yes/no gates before consequential actions
5. Manual Review — surfacing intermediate artifacts (drafts, plans) for human editing
6. Feedback Loops — collecting structured human feedback and routing based on it


All of these depend on checkpointing. without a checkpointer , there is no pause/resume-- The graph has no memory of where it stopped.

#### Core Mental Model
``` markdown
Graph running →  reaches interrupt node  →  PAUSED (state saved to checkpointer)
                                                     ↓
                                          Human reviews state
                                                     ↓
                          Human approves / edits state / rejects
                                                     ↓
              graph.invoke(None, config)  →  RESUMED from checkpoint
```

The key insight: when a graph is interrupted, you do NOT re-invoke with the original input. You re-invoke with None as the input and the same thread_id in the config. LangGraph reads the saved checkpoint and picks up exactly where it left off.


In [8]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("groq_api_key")
model_name = os.getenv("groq_model_name")

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated
import operator

llm = ChatGroq(
    api_key = os.getenv("groq_api_key"),
    model_name = model_name
)
print(llm)

metadata={'versions': {'langchain-core': '1.4.6', 'langchain': '1.3.8'}} output_version=None profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True} client=<groq.resources.chat.completions.Completions object at 0x0000021BA2296A10> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021BA233D210> model_name='llama-3.3-70b-versatile' model_kwargs={} groq_api_key=SecretStr('**********') groq_api_base=None groq_proxy=None


### Part 1 — Interrupt
1. What is interrupt_before?

```interrupt_before``` is a compile-time option that tells LangGraph: "before executing this node, pause and wait for human input." The node itself is NOT called until the human resumes.

``` python
graph = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["node_name"]   # list of node names to pause before
)
```

we can also use interrupt_after to pause AFTER a node runs (useful when you want to inspect what the node produced before continuing).

``` python
graph = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_after=["node_name"]    # pause after node runs, before next node
)
```







In [19]:
### working example

from langgraph.graph import StateGraph, START,END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict
from langgraph.types import interrupt, Command

class State(TypedDict):
    task: str
    plan: str
    approved: bool
    result: str

def plan_node(state: State) -> dict:
    """LLM generates a plan for the task."""
    plan = f"Plan for '{state['task']}': Step 1 → Step 2 → Step 3"
    print(f"[plan_node] Generated plan: {plan}")
    return {"plan": plan}

def execute_node(state: State) -> dict:
    """Executes the approved plan."""
    print(f"[execute_node] Executing: {state['plan']}")
    return {"result": f"Completed: {state['plan']}"}

builder = StateGraph(State)
builder.add_node("plan", plan_node)
builder.add_node("execute", execute_node)
builder.set_entry_point("plan")
builder.add_edge("plan", "execute")
builder.add_edge("execute", END)

checkpointer = MemorySaver()

graph = builder.compile(
    checkpointer=checkpointer,
    interrupt_before=["execute"]   # ← pause BEFORE execute runs
)
# --- Run 1: Graph pauses before execute ---
config = {"configurable": {"thread_id": "thread-001"}}
initial_state = {"task": "Send marketing email to all users", "approved": False}
result = graph.invoke(initial_state, config)
print("\n--- Graph paused. Current state: ---")
print(result)

# --- Human reviews the plan here ---
print("\nHuman reviewing plan...")
print(f"Plan: {result['plan']}")

# --- Run 2: Resume execution ---
print("\nHuman approved. Resuming...")
final = graph.invoke(None, config)   # ← None input, same thread_id
print("\n--- Final state: ---")
print(final)




[plan_node] Generated plan: Plan for 'Send marketing email to all users': Step 1 → Step 2 → Step 3

--- Graph paused. Current state: ---
{'task': 'Send marketing email to all users', 'plan': "Plan for 'Send marketing email to all users': Step 1 → Step 2 → Step 3", 'approved': False}

Human reviewing plan...
Plan: Plan for 'Send marketing email to all users': Step 1 → Step 2 → Step 3

Human approved. Resuming...
[execute_node] Executing: Plan for 'Send marketing email to all users': Step 1 → Step 2 → Step 3

--- Final state: ---
{'task': 'Send marketing email to all users', 'plan': "Plan for 'Send marketing email to all users': Step 1 → Step 2 → Step 3", 'approved': False, 'result': "Completed: Plan for 'Send marketing email to all users': Step 1 → Step 2 → Step 3"}


In [13]:
# new practical implementation

class CodeState(TypedDict):
    user_request: str
    generated_code: str
    human_feedback: str

# node 1
def generate_code(state: CodeState):

    prompt = f"""
    Write Python code.

    Requirement:
    {state["user_request"]}
    """

    response = llm.invoke(prompt)

    return {
        "generated_code": response.content
    }

# node 2. Human review (interrupt)
def human_review(state: CodeState):

    feedback = interrupt(
        {
            "generated_code": state["generated_code"],
            "message": "Approve this code or provide modifications."
        }
    )

    return {
        "human_feedback": feedback
    }

# decision function

def review_decision(state: CodeState):

    feedback = state["human_feedback"].lower()

    if feedback == "approve":
        return "approved"

    return "modify"

# node 3 : modufy the code

def modify_code(state: CodeState):
    prompt = f"""
    Here is the existing code:

    {state["generated_code"]}

    Human feedback:

    {state["human_feedback"]}

    Modify the code accordingly.
    """

    response = llm.invoke(prompt)

    return {
        "generated_code": response.content
    }

builder = StateGraph(CodeState)

builder.add_node("generate", generate_code)
builder.add_node("review", human_review)
builder.add_node("modify", modify_code)

builder.add_edge(START, "generate")
builder.add_edge("generate", "review")

builder.add_conditional_edges(
    "review",
    review_decision,
    {
        "approved": END,
        "modify": "modify"
    }
)

builder.add_edge("modify", "review")

# memory

memory = MemorySaver()

graph = builder.compile(
    checkpointer=memory
)


In [14]:
config = {
    "configurable": {
        "thread_id": "thread-1"
    }
}

result = graph.invoke(
    {
        "user_request": "Write a Python function to sort a list."
    },
    config=config
)

print(result)

{'user_request': 'Write a Python function to sort a list.', 'generated_code': '**Sorting a List in Python**\n================================\n\nHere\'s a simple Python function that uses the built-in `sorted()` function to sort a list.\n\n```python\ndef sort_list(input_list):\n    """\n    Sorts a list in ascending order.\n\n    Args:\n        input_list (list): The list to be sorted.\n\n    Returns:\n        list: The sorted list.\n    """\n    return sorted(input_list)\n\n# Example usage:\nnumbers = [64, 34, 25, 12, 22, 11, 90]\nprint("Original list:", numbers)\nprint("Sorted list:", sort_list(numbers))\n```\n\n**Output:**\n```\nOriginal list: [64, 34, 25, 12, 22, 11, 90]\nSorted list: [11, 12, 22, 25, 34, 64, 90]\n```\n\nIf you want to implement a sorting algorithm from scratch, here\'s an example using the QuickSort algorithm:\n\n```python\ndef quicksort(arr):\n    """\n    Sorts a list using the QuickSort algorithm.\n\n    Args:\n        arr (list): The list to be sorted.\n\n    

In [15]:
graph.invoke(
    Command(
        resume="Use merge sort instead of bubble sort."
    ),
    config=config
)

{'user_request': 'Write a Python function to sort a list.',
 'generated_code': '## Sorting a List in Python\n================================\n\nHere\'s a simple Python function that uses the built-in `sorted()` function to sort a list.\n\n```python\ndef sort_list(input_list):\n    """\n    Sorts a list in ascending order.\n\n    Args:\n        input_list (list): The list to be sorted.\n\n    Returns:\n        list: The sorted list.\n    """\n    return sorted(input_list)\n\n# Example usage:\nnumbers = [64, 34, 25, 12, 22, 11, 90]\nprint("Original list:", numbers)\nprint("Sorted list:", sort_list(numbers))\n```\n\n## Output:\n```\nOriginal list: [64, 34, 25, 12, 22, 11, 90]\nSorted list: [11, 12, 22, 25, 34, 64, 90]\n```\n\nIf you want to implement a sorting algorithm from scratch, here\'s an example using the Merge Sort algorithm:\n\n```python\ndef merge_sort(arr):\n    """\n    Sorts a list using the Merge Sort algorithm.\n\n    Args:\n        arr (list): The list to be sorted.\n\n  

In [16]:
graph.invoke(
    Command(
        resume="Add comments and type hints."
    ),
    config=config
)

{'user_request': 'Write a Python function to sort a list.',
 'generated_code': '### Modified Code\n\nHere is the modified code with added comments and type hints:\n\n```python\ndef sort_list(input_list: list) -> list:\n    """\n    Sorts a list in ascending order using the built-in sorted() function.\n\n    Args:\n        input_list (list): The list to be sorted.\n\n    Returns:\n        list: The sorted list.\n    """\n    # Use the built-in sorted() function to sort the list\n    return sorted(input_list)\n\n# Example usage:\nnumbers = [64, 34, 25, 12, 22, 11, 90]\nprint("Original list:", numbers)\nprint("Sorted list:", sort_list(numbers))\n\n\ndef merge_sort(arr: list) -> list:\n    """\n    Sorts a list using the Merge Sort algorithm.\n\n    Args:\n        arr (list): The list to be sorted.\n\n    Returns:\n        list: The sorted list.\n    """\n    # Base case: If the list has one or zero elements, it is already sorted\n    if len(arr) <= 1:\n        return arr\n\n    # Divide t

In [17]:
graph.invoke(
    Command(
        resume="approve"
    ),
    config=config
)

{'user_request': 'Write a Python function to sort a list.',
 'generated_code': '### Modified Code\n\nHere is the modified code with added comments and type hints:\n\n```python\ndef sort_list(input_list: list) -> list:\n    """\n    Sorts a list in ascending order using the built-in sorted() function.\n\n    Args:\n        input_list (list): The list to be sorted.\n\n    Returns:\n        list: The sorted list.\n    """\n    # Use the built-in sorted() function to sort the list\n    return sorted(input_list)\n\n# Example usage:\nnumbers = [64, 34, 25, 12, 22, 11, 90]\nprint("Original list:", numbers)\nprint("Sorted list:", sort_list(numbers))\n\n\ndef merge_sort(arr: list) -> list:\n    """\n    Sorts a list using the Merge Sort algorithm.\n\n    Args:\n        arr (list): The list to be sorted.\n\n    Returns:\n        list: The sorted list.\n    """\n    # Base case: If the list has one or zero elements, it is already sorted\n    if len(arr) <= 1:\n        return arr\n\n    # Divide t

#### What happens step by step

* ```graph.invoke(initial_state, config)``` — plan_node runs, state is saved, graph pauses before execute

* You inspect result — the plan is visible

* graph.invoke(None, config) — LangGraph reads the checkpoint, skips plan_node (already done), runs execute_node

#### 1.3 interrupt_after Example

Use interrupt_after when the node should run first, and you want to inspect or modify what it produced.

In [20]:
graph = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_after=["plan"]   # plan runs, THEN pause
)

config = {"configurable": {"thread_id": "thread-002"}}
result = graph.invoke({"task": "Deploy to production"}, config)

# plan_node has already run — we can see the plan
print(f"Plan generated: {result['plan']}")

# Human can modify the plan via update_state (covered in Part 3)
# Then resume
final = graph.invoke(None, config)

[plan_node] Generated plan: Plan for 'Deploy to production': Step 1 → Step 2 → Step 3
Plan generated: Plan for 'Deploy to production': Step 1 → Step 2 → Step 3
[execute_node] Executing: Plan for 'Deploy to production': Step 1 → Step 2 → Step 3


In [21]:
# 1.4 Checking Graph State After Interrupt

# Use get_state() to inspect what the graph knows at the paused checkpoint.

snapshot = graph.get_state(config)

print(snapshot.values)          # the full state dict
print(snapshot.next)            # which node(s) will run on resume
print(snapshot.metadata)        # step count, source info

{'task': 'Deploy to production', 'plan': "Plan for 'Deploy to production': Step 1 → Step 2 → Step 3", 'result': "Completed: Plan for 'Deploy to production': Step 1 → Step 2 → Step 3"}
()
{'source': 'loop', 'step': 2, 'parents': {}}


```snapshot.next``` is your clearest signal that the graph is paused — it contains the name(s) of the node(s) waiting to run.



In [22]:
# Pattern to check if graph is paused
snapshot = graph.get_state(config)
if snapshot.next:
    print(f"Graph is paused. Waiting on: {snapshot.next}")
else:
    print("Graph has completed.")

Graph has completed.


### Resume

interrupt pauses. Resume Restart from the paused state.

1. The Resume Mechanic

Resume is simply re-invoking the graph with None as input and the same thread config.

``` python
# Pause
graph.invoke(initial_input, config)

# Resume
graph.invoke(None, config)
```
LangGraph does the rest — reads the checkpoint, knows which node to run next, and continues.

2. Resuming Multiple Times

A graph can be interrupted and resumed multiple times across its lifetime — once per interrupt_before node in the execution path.




In [23]:
## Example
class PipelineState(TypedDict):
    data: str
    cleaned_data: str
    analysis: str
    report: str

def clean_node(state): 
    return {"cleaned_data": f"cleaned({state['data']})"}

def analyze_node(state): 
    return {"analysis": f"analysis of {state['cleaned_data']}"}

def report_node(state): 
    return {"report": f"Report: {state['analysis']}"}

builder = StateGraph(PipelineState)
builder.add_node("clean", clean_node)
builder.add_node("analyze", analyze_node)
builder.add_node("report", report_node)
builder.set_entry_point("clean")
builder.add_edge("clean", "analyze")
builder.add_edge("analyze", "report")
builder.add_edge("report", END)

graph = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["analyze", "report"]  # two interrupts
)

config = {"configurable": {"thread_id": "pipeline-001"}}

# Run 1: clean runs, pauses before analyze
s1 = graph.invoke({"data": "raw_data"}, config)
print(f"Paused 1. Next: {graph.get_state(config).next}")  # ('analyze',)

# Run 2: analyze runs, pauses before report
s2 = graph.invoke(None, config)
print(f"Paused 2. Next: {graph.get_state(config).next}")  # ('report',)

# Run 3: report runs, graph completes
s3 = graph.invoke(None, config)
print(f"Done. Next: {graph.get_state(config).next}")      # ()
print(s3)

Paused 1. Next: ('analyze',)
Paused 2. Next: ('report',)
Done. Next: ()
{'data': 'raw_data', 'cleaned_data': 'cleaned(raw_data)', 'analysis': 'analysis of cleaned(raw_data)', 'report': 'Report: analysis of cleaned(raw_data)'}


3. Resuming with Stream

You can also resume using stream() for real-time visibility into what each node produces as it runs.

``` python
# Resume with streaming output
for chunk in graph.stream(None, config, stream_mode="values"):
    print(chunk)
```


